# 02I — Mutation classes


In [15]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import fisher_exact, chi2_contingency, mannwhitneyu, kruskal, spearmanr, ks_2samp, entropy, norm
from IPython.display import display

PROCESSED = Path('../../data/processed')
df = pd.read_csv(PROCESSED / 'DMD_variants_annotated.csv')

print('Loaded:', PROCESSED / 'DMD_variants_annotated.csv')
print('Shape:', df.shape)


Loaded: ../../data/processed/DMD_variants_annotated.csv
Shape: (11308, 30)


In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = None
for rel in ('../..', '..', '.'):
    cand = (Path.cwd() / rel).resolve()
    if (cand / 'src' / 'utils.py').exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('project root with src/utils.py not found')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    add_consistency_flags,
    bh_adjust,
    chi2_table,
    ecdf_xy,
    fisher_bool,
    frame_mismatch,
    kruskal_group,
    mann_whitney_bool,
    mut_cons_mismatch,
    odds_ratio_ci,
    path_cons_mismatch,
    prepare_eda_dataframe,
    spearman_xy,
)

d = prepare_eda_dataframe(df)

print('Prepared shape:', d.shape)


Prepared shape: (11308, 59)


## 1. 📊 mutation_type distribution


In [17]:
tbl = d['mutation_type'].value_counts().reset_index()
tbl.columns = ['mutation_type', 'count']
display(tbl)
fig = px.bar(tbl, x='mutation_type', y='count', title='Mutation type distribution')
fig.show()


,mutation_type,count
0,missense,3328
1,large_deletion,1701
2,splice,940
3,nonsense,791
4,frameshift,754


## 2. 📊 mutation_type with phenotype


In [18]:
tmp = d[['mutation_type', 'phenotype']].dropna()
fig = px.histogram(tmp, x='mutation_type', color='phenotype', barmode='stack', title='Mutation type x phenotype')
fig.show()


## 3. 🧪 χ² mutation_type ~ phenotype


In [19]:
tab, chi2, p, dof = chi2_table(d, 'mutation_type', 'phenotype')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


phenotype,BMD,DMD
mutation_type,,
frameshift,53,525
large_deletion,26,1051
missense,423,2344
nonsense,116,551
splice,68,706


,value
chi2,1.552106e+02
dof,4.000000e+00
p_value,1.555584e-32


## 4. 📊 mutation_type with pathogenic


In [20]:
tmp = d[['mutation_type', 'pathogenic']].dropna()
fig = px.histogram(tmp, x='mutation_type', color='pathogenic', barmode='stack', title='Mutation type x pathogenic')
fig.show()


## 5. 🧪 χ² mutation_type ~ pathogenic


In [21]:
tab, chi2, p, dof = chi2_table(d, 'mutation_type', 'pathogenic')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


pathogenic,False,True
mutation_type,,
frameshift,6,748
large_deletion,281,1420
missense,3253,75
nonsense,184,607
splice,541,399


,value
chi2,4724.286243
dof,4.000000
p_value,0.000000


## 6. 📊 mutation_type with frame


In [22]:
tmp = d[['mutation_type', 'frame']].dropna()
fig = px.histogram(tmp, x='mutation_type', color='frame', barmode='stack', title='Mutation type x frame')
fig.show()


## 7. 🧪 χ² mutation_type ~ frame


In [23]:
tab, chi2, p, dof = chi2_table(d, 'mutation_type', 'frame')
display(tab)
display(pd.Series({'chi2': chi2, 'dof': dof, 'p_value': p}).to_frame('value'))


frame,in-frame,out-of-frame
mutation_type,,
frameshift,0,754
large_deletion,0,1701
missense,3328,0
nonsense,0,791
splice,0,940


,value
chi2,7514.0
dof,4.0
p_value,0.0


## 8. 📊 mutation_type vs exon


In [24]:
tmp = d[['mutation_type', 'exon_num']].dropna()
fig = px.box(tmp, x='mutation_type', y='exon_num', points='outliers', title='Mutation type vs exon')
fig.show()


## 9. 📊 mutation_type vs domain


In [25]:
top_dom = d['domain_clean'].value_counts().head(12).index
tmp = d[d['domain_clean'].isin(top_dom)]
heat = pd.crosstab(tmp['mutation_type'], tmp['domain_clean'])
fig = px.imshow(heat, aspect='auto', color_continuous_scale='Blues', title='Mutation type vs domain')
fig.show()


## 10. 📊 mutation_type entropy 📖


In [26]:
p = d['mutation_type'].value_counts(normalize=True)
ent = entropy(p.values, base=2)
display(pd.Series({'n_types': int(p.shape[0]), 'entropy_bits': float(ent)}).to_frame('value'))


,value
n_types,5.000000
entropy_bits,2.055451


## 11. 📊 mutation_type vs interval_length


In [27]:
tmp = d[['mutation_type', 'interval_length_num']].dropna()
fig = px.box(tmp, x='mutation_type', y='interval_length_num', points='outliers', title='Mutation type vs interval_length')
fig.update_yaxes(type='log')
fig.show()


## 12. 🧪 Kruskal mutation_type ~ interval_length


In [28]:
tmp = d[d['mutation_type'].isin(d['mutation_type'].value_counts()[lambda s: s >= 20].index)]
names, groups, stat, p = kruskal_group(tmp, 'mutation_type', 'interval_length_num')
display(pd.Series({'n_groups': len(names), 'kruskal_h': stat, 'p_value': p}).to_frame('value'))


,value
n_groups,4.000000e+00
kruskal_h,1.195510e+03
p_value,6.907799e-259
